# Stat 220, Unit 2 Homework: A Map of Models

`campus_cafe.csv` is 700 days at a campus coffee shop: `temp_f`, `exam_week`,
`promo`, `foot_traffic`, `drinks_sold`, `revenue`, and `sold_out`.

```python
import pandas as pd, numpy as np
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

cafe = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/campus_cafe.csv")
cafe.head()
```

In [18]:
import pandas as pd, numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

cafe = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/campus_cafe.csv")
cafe.head()


,day_of_week,temp_f,exam_week,promo,foot_traffic,drinks_sold,revenue,sold_out
0,Fri,64.1,0,1,908,20,73.00,1
1,Mon,40.1,0,0,877,21,79.75,0
2,Mon,38.0,1,0,1243,27,137.08,1
3,Thu,50.9,0,0,1442,35,140.86,0
4,Tue,68.6,0,0,1060,23,104.40,0


**Problem 1.** *Reading a situation.* No computer.

Part a. The owner wants to know how many drinks to prepare tomorrow, given the forecast temperature. What type is `y` here, and which model family would you name?

y is drinks_sold, which is a quantitative/count outcome. I would use a regression model, such as linear regression, because the owner wants to predict how many drinks will be sold based on temperature.


Part b. The owner wants to know whether running a promotion actually raises revenue, since she is deciding whether to keep doing it. What type is `y`, which family, and what makes this a different job from part a?

y is revenue, which is a quantitative outcome, so a regression model would be appropriate. This is different from part a because the owner is not just trying to predict revenue; she wants to know whether the promotion actually causes an increase in revenue



Part c. The owner wants the chance they run out of an item on a given day. What type is `y`, and which family?

y is sold_out, which is a yes/no outcome. I would use a logistic regression model because logistic regression models the probability of an outcome occurring.



Part d. A classmate says to just use a random forest for all three. A forest would run on all three without complaining. Say which of the three you would refuse to use it for, and why.

I would refuse to use a random forest for part b, because the goal is to determine whether the promotion actually raises revenue. A random forest is mainly a prediction tool and does not provide the same kind of  coefficients or p-values that can help with an inference question.



**Problem 2.** *What happens if you ignore the type of `y`.*

Part a. `sold_out` is a yes/no column. Describe what the linear model produced that the logistic model did not, and say why that is a problem.

In [19]:
# the right model for a yes/no outcome
p_ok = smf.logit("sold_out ~ drinks_sold", data=cafe).fit(disp=0).predict()

# the same outcome, forced into a linear model
p_bad = smf.ols("sold_out ~ drinks_sold", data=cafe).fit().predict()

print(f"logistic gives values from {p_ok.min():.3f} to {p_ok.max():.3f}")
print(f"linear   gives values from {p_bad.min():.2f} to {p_bad.max():.2f}")
print("linear values that are not possible probabilities:",
      int(((p_bad < 0) | (p_bad > 1)).sum()), "out of", len(p_bad))

logistic gives values from 0.026 to 0.971
linear   gives values from -0.19 to 1.15
linear values that are not possible probabilities: 25 out of 700


The linear model produced some predicted values that were less than 0 or greater than 1, while the logistic model kept the predictions between 0 and 1. This is a problem because sold_out is a yes/no outcome, so its predicted probability must be between 0 and 1. The Linear model ad 25 probabilites that were not possible in reality. 



Part b. `drinks_sold` is a count. Name the probability model you would use for it, and say in one sentence what would go wrong with an ordinary linear model there.

For drinks_sold, I would use a Poisson probability model because drinks sold is a count. An ordinary linear model can produce impossible negative predictions and does not properly model the distribution of count data.



**Problem 3.** *What a model can and cannot hand back.*

Part a. List the quantities the forest could not produce, and explain why not. Your answer should use the word distribution.

In [20]:
X = cafe[["drinks_sold", "exam_week", "promo", "temp_f"]]
reg = LinearRegression().fit(X, cafe["revenue"])
forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(X, cafe["revenue"])
sm_reg = smf.ols("revenue ~ drinks_sold + exam_week + promo + temp_f", data=cafe).fit()

for name, model in [("regression", sm_reg), ("forest", forest)]:
    have = [a for a in ["pvalues", "conf_int", "aic"] if hasattr(model, a)]
    print(f"{name:<12} can give you: {have if have else 'none of them'}")

regression   can give you: ['pvalues', 'conf_int', 'aic']
forest       can give you: none of them


The forest could not produce p-values, confidence intervals, or AIC. These quantities come from a statistical model with assumptions about the distribution of the data and are used for inference and model comparison.



Part b. Go back to the three jobs in Problem 1. Which one of them could a forest not do at all, given what you just saw?

The forest could not do part b of Problem 1 in the same way because that question is asking whether the promotion actually raises revenue. The forest can predict revenue, but it does not provide the inference quantities needed to determine the effect of the promotion.



**Problem 4.** *Is model A better than model B?*

Part a. These three models are compared by AIC. Say which one you would keep, and what it means that one of the additions made AIC go up.

In [21]:
for f in ["revenue ~ drinks_sold",
          "revenue ~ drinks_sold + exam_week",
          "revenue ~ drinks_sold + exam_week + promo"]:
    print(f"{f:<48} AIC = {smf.ols(f, data=cafe).fit().aic:8.1f}")

revenue ~ drinks_sold                            AIC =   5492.8
revenue ~ drinks_sold + exam_week                AIC =   5442.6
revenue ~ drinks_sold + exam_week + promo        AIC =   5444.3


I would keep the model with the lowest AIC: revenue ~ drinks_sold + exam_week. If adding a variable causes AIC to go up, it means that the added variable did not improve the model enough to justify the extra complexity according to AIC. 



Part b. Say which model you would ship, and whether the result surprises you.

In [22]:
folds = KFold(5, shuffle=True, random_state=0)   # one fold object, used for both
for name, mod in [("regression", LinearRegression()),
                  ("forest", RandomForestRegressor(n_estimators=300, random_state=0))]:
    mse = -cross_val_score(mod, X, cafe["revenue"], cv=folds,
                           scoring="neg_mean_squared_error").mean()
    print(f"  {name:<12} cross-validated MSE {mse:7.1f}   RMSE {np.sqrt(mse):5.1f} dollars")

  regression   cross-validated MSE   140.2   RMSE  11.8 dollars
  forest       cross-validated MSE   171.9   RMSE  13.1 dollars


I would ship the regression model because it has the lower cross-validated RMSE/MSE. The result does not necessarily surprise me because random forests can capture nonlinear relationships and interactions that ordinary linear regression cannot so it makes sense that the numbers are so close between them.


Part c. You could compare the three models in part a with AIC, but you could not use AIC for the comparison in part b. Explain why in two sentences.

AIC can be used for the three models in part a because they are all statistical regression models that provide a likelihood and AIC. The comparison in part b uses cross-validation and prediction error, as the random forest does not provide AIC.



**Problem 5.** *Putting it together.* No computer.

Part a. The owner reads that a forest predicted revenue almost as well as the regression, and asks whether she should use the forest to decide about promotions. Answer her in three or four sentences.

I would not use the forest to decide whether promotions actually raise revenue just because it predicts revenue almost as well as the regression. A forest is useful for prediction, but it does not give the same inference information that can help determine the effect of a promotion. If the goal is to make a decision about whether promotions cause higher revenue, I would use a model designed for inference so we can get the most accurate prediction.

